# GP-Based Soay Sheep IBM

Fit GP vital rates to the 1,000-row baseline sample, generate a short pooled IBM dataset, and construct the fitted GP IPM. Recruitment is intercept-only and offspring sex probability is fixed at 0.5.

In [8]:
import numpy as np
import pandas as pd
import gpflow
import tensorflow as tf
from scipy.special import expit, logit
from ipm_utils import calculate_ipm_metrics, make_gp_ipm, predict_f_mean, predict_y_mean, sample_f, sigmoid

In [9]:
baseline = pd.read_csv("baseline_ibm_data_1000.csv")
growth = baseline[["z", "z1"]].dropna()
survival = baseline[["z", "Surv"]].dropna()
reproduction = baseline[["z", "Repr"]].dropna()
recruitment = baseline["Recr"].dropna().to_numpy()
recruit_size = baseline[["z", "Rcsz"]].dropna()

X_growth = growth[["z"]].to_numpy(np.float64)
Y_growth = growth[["z1"]].to_numpy(np.float64)
X_surv = survival[["z"]].to_numpy(np.float64)
Y_surv = survival[["Surv"]].to_numpy(np.float64)
X_repr = reproduction[["z"]].to_numpy(np.float64)
Y_repr = reproduction[["Repr"]].to_numpy(np.float64)
X_rcsz = recruit_size[["z"]].to_numpy(np.float64)
Y_rcsz = recruit_size[["Rcsz"]].to_numpy(np.float64)

In [10]:
model_growth = gpflow.models.GPR((X_growth, Y_growth), gpflow.kernels.SquaredExponential())
model_surv = gpflow.models.VGP((X_surv, Y_surv), gpflow.kernels.SquaredExponential(), gpflow.likelihoods.Bernoulli(invlink=tf.sigmoid))
model_repr = gpflow.models.VGP((X_repr, Y_repr), gpflow.kernels.SquaredExponential(), gpflow.likelihoods.Bernoulli(invlink=tf.sigmoid))
model_rcsz = gpflow.models.GPR((X_rcsz, Y_rcsz), gpflow.kernels.SquaredExponential())
optimizer = gpflow.optimizers.Scipy()
for model in [model_growth, model_surv, model_repr, model_rcsz]:
    optimizer.minimize(model.training_loss, model.trainable_variables)

p_recr = np.clip(recruitment.mean(), 1e-6, 1 - 1e-6)
params = {"growth_noise": float(np.sqrt(model_growth.likelihood.variance.numpy())), 
          "rcsz_noise": float(np.sqrt(model_rcsz.likelihood.variance.numpy())), 
          "recr_int": float(logit(p_recr)), 
          "female_prob": 0.5}

funcs = {"growth": model_growth, 
         "surv": model_surv, 
         "repr": model_repr, 
         "rcsz": model_rcsz}

In [11]:
rng = np.random.default_rng(562)
tf.random.set_seed(562)
initial_mean = predict_f_mean(model_rcsz, np.array([3.2]))[0]
z = rng.normal(initial_mean, params["rcsz_noise"], 250)
frames = []
year = 1
while year < 5 and 0 < len(z) < 5000:
    n = len(z)
    surv = rng.binomial(1, np.clip(sigmoid(sample_f(model_surv, z)), 0, 1))
    survived = surv == 1

    z1 = np.full(n, np.nan)
    z1[survived] = sample_f(model_growth, z[survived]) + params["growth_noise"] * rng.normal(size=survived.sum())
    repr_ = np.full(n, np.nan)
    repr_[survived] = rng.binomial(1, np.clip(sigmoid(sample_f(model_repr, z[survived])), 0, 1))
    reproduced = survived & (repr_ == 1)

    sex = np.full(n, np.nan)
    sex[reproduced] = rng.binomial(1, 0.5, reproduced.sum())
    female = reproduced & (sex == 1)

    recr = np.full(n, np.nan)
    recr[female] = rng.binomial(1, p_recr, female.sum())
    recruited = female & (recr == 1)
    
    rcsz = np.full(n, np.nan)
    rcsz[recruited] = sample_f(model_rcsz, z[recruited]) + params["rcsz_noise"] * rng.normal(size=recruited.sum())
    frames.append(pd.DataFrame({"z": z, "Surv": surv, "z1": z1, "Repr": repr_, "Sex": sex, "Recr": recr, "Rcsz": rcsz, "yr": year}))
    z = np.concatenate([rcsz[recruited], z1[survived]])
    year += 1

sim_data = pd.concat(frames, ignore_index=True)
if len(sim_data) > 1000:
    sim_data = sim_data.sample(1000, replace=False, random_state=562).reset_index(drop=True)
sim_data.to_csv("gp_ibm_dataset.csv", index=False)
print(f"Pooled Observations: {len(sim_data)}")

Pooled Observations: 760


In [12]:
L, U, n_mesh = 1.6, 3.7, 250
gp_ipm = make_gp_ipm(n_mesh, L, U, params, funcs, correction=True)
mesh = gp_ipm["mesh_points"]
gp_metrics = calculate_ipm_metrics(gp_ipm["K"], predict_y_mean(model_repr, mesh))
np.savez_compressed("true_gp_metrics.npz", **gp_metrics)
print(f"GP IBM Lambda: {gp_metrics['lambda']:.6f}")

GP IBM Lambda: 1.013638
